# Pipeline evaluation

Three-layer eval across every recording in the eval tree (`~/datasets/eval/`).
Each layer gracefully degrades for recordings that lack inputs (e.g. SQUIM
always runs; intrusive metrics only when oracle audio exists).

**Layout per recording**:
```
~/datasets/eval/<dataset>/<recording_id>/
  mixture.wav
  reference/
    speaker_A.wav  speaker_B.wav        # oracle audio (optional)
    speaker_A.txt  speaker_B.txt        # GT transcripts (required for L3)
    diarization.rttm                    # GT-derived (required for L1)
  pipeline/         pipeline_nosep/   pipeline_noenh/
    diarization.json, stream_A.wav, stream_B.wav, transcript_*.txt, ...
```

**Producers**: `scripts/prepare_eval_references.py` (everything under `reference/`),
`scripts/run_pipeline_on_recording.py` (the three `pipeline*/` dirs). This notebook
is read-only against that tree.

In [ ]:
import os, sys
from pathlib import Path

PROJECT_ROOT = Path.cwd() if Path('asr_pipeline').is_dir() else Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from IPython.display import display
import pandas as pd
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 20)
pd.set_option('display.float_format', lambda x: f'{x:.2f}')

from asr_pipeline.eval import (
    walk_eval_tree, evaluate_recording, evaluate_many,
    inventory, summarize_layer1, summarize_layer2_intrusive,
    summarize_layer2_squim, summarize_layer3,
)

EVAL_ROOT = Path('~/datasets/eval').expanduser()
print(f'EVAL_ROOT: {EVAL_ROOT}')
print(f'exists:    {EVAL_ROOT.exists()}')

## Inventory

What's available per recording. Use this to spot recordings that won't get
scored on certain layers (e.g. no oracle audio → only non-intrusive SQUIM)
or that haven't had the pipeline run yet.

In [ ]:
# Filter knob: restrict to one dataset by name (None = walk everything).
DATASET = None   # 'clarin' | 'libricss' | None

recordings = list(walk_eval_tree(EVAL_ROOT, dataset=DATASET))
print(f'discovered {len(recordings)} recording(s)')

# Inventory uses ScoreCard, so we build minimal ones (no eval yet).
from asr_pipeline.eval.run import ScoreCard
stub_cards = [ScoreCard(recording=r, layer1=None, layer2=None, layer3=None) for r in recordings]
display(inventory(stub_cards))

## Evaluate

Run all three layers per recording. Layer 2 SQUIM loads a model on GPU
(~few seconds); `evaluate_many` loads it once and reuses across the sweep.

In [ ]:
# Knobs.
ONLY_WITH_PIPELINE = True   # skip recordings that haven't been run yet
DER_COLLAR = 0.0
TCP_COLLAR_S = 5.0

to_eval = [r for r in recordings if (not ONLY_WITH_PIPELINE) or r.pipeline_dir is not None]
print(f'evaluating {len(to_eval)} recording(s)...')
scores = evaluate_many(
    to_eval, der_collar=DER_COLLAR, tcp_collar_s=TCP_COLLAR_S,
)
print('done.')

## Layer 1 — DER

Hypothesis: stage-1 pyannote diarization (the pipeline's own output, what
drove routing/separation/assembly downstream). Reference: the RTTM derived
from GT transcripts.

Reads as: low `der_pct` is good. `miss_pct` high → pipeline missed turns;
`fa_pct` high → pipeline detected speech in silence; `conf_pct` high →
speaker swaps.

In [ ]:
df1 = summarize_layer1(scores)
display(df1)
if not df1.empty and df1['der_pct'].notna().any():
    print()
    print(f"mean DER (across {df1['der_pct'].notna().sum()} recordings):\n"
          f"  DER  = {df1['der_pct'].mean():.2f}%\n"
          f"  miss = {df1['miss_pct'].mean():.2f}%   "
          f"FA = {df1['fa_pct'].mean():.2f}%   "
          f"conf = {df1['conf_pct'].mean():.2f}%")

## Layer 2 — audio quality

Two tables:

- **Intrusive** (SI-SDR / PESQ-WB / STOI vs oracle audio + improvement
  over the mono-mix baseline). Only computable when oracle audio is
  present.
- **Non-intrusive SQUIM** (per-stream estimate of STOI / PESQ / SI-SDR
  without a reference). Always available.

In [ ]:
df2i = summarize_layer2_intrusive(scores)
if df2i.empty:
    print('no intrusive scores — no oracle audio for any evaluated recording')
else:
    display(df2i)

In [ ]:
df2s = summarize_layer2_squim(scores)
display(df2s)

## Layer 3 — WER ablation

Four numbers per recording, telling the ablation story:

| column          | what it measures |
| --------------- | ---------------- |
| `mixture_orc`   | ORC-WER of single-stream Whisper on the raw mixture (baseline) |
| `no_enh_cpwer`  | cpWER with enhancement disabled (separator alone) |
| `no_sep_cpwer`  | cpWER with separation disabled (enhancement + diarization alone) |
| `full_cpwer`    | cpWER of the full pipeline |

We expect `mixture_orc ≥ no_X_cpwer ≥ full_cpwer` for the ablation to tell
the expected story. A row that breaks the ordering is the interesting
finding.

In [ ]:
df3 = summarize_layer3(scores)
display(df3)

## When numbers look wrong

- **L1 high `miss_pct` but pipeline transcripts look fine** — stage-1
  diarization missed some speech but separation recovered; check
  `pipeline/diarization.json` against `reference/diarization.rttm` for the
  recording.
- **L2 intrusive ≈ baseline** — the pipeline didn't improve over the
  raw mixture for that speaker. Look at SQUIM scores per stream too — if
  SQUIM is fine, the intrusive metric is probably picking up
  permutation/alignment mismatches.
- **L3 modes inverted (full worse than no_sep)** — separator made things
  worse for that recording. Listen to `pipeline/stream_*.wav` vs
  `pipeline_nosep/stream_*.wav` to confirm.
- **One recording dominates the mean** — switch to `df.median()` or
  filter outliers by hand; the eval tree is small enough that a single
  bad recording skews the average noticeably.